# 01 — Data & Embeddings

**Does one thing, nothing more:**
1. Downloads MovieLens 1M
2. Converts each movie to a text description
3. Encodes with Sentence-BERT
4. Caches the result

**output:** `data/movies.pkl` and `data/embeddings.npy`

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Project imports
from src.config import set_seeds, DATA_DIR, GLOBAL_SEED
from src.data_loader import MovieLensDataset
from src.logging_setup import setup_logging

# Standard library
import json
from datetime import datetime

# Third-party
import numpy as np
import pandas as pd

# Setup
logger = setup_logging(__name__)
set_seeds()

print("✓ Imports successful")
print(f"✓ Project root: {project_root}")
print(f"✓ Data directory: {DATA_DIR}")
print(f"✓ Random seed: {GLOBAL_SEED}")

imports OK


### Step 1 — Download MovieLens 1M

In [2]:
def download_movielens(data_dir: Path):
    if (data_dir / 'movies.dat').exists():
        print('Already downloaded.')
        return
    url = 'https://files.grouplens.org/datasets/movielens/ml-1m.zip'
    print('Downloading ...')
    r = requests.get(url)
    z = zipfile.ZipFile(io.BytesIO(r.content))
    for name in z.namelist():
        if name.endswith('.dat'):
            fname = Path(name).name
            with z.open(name) as src, open(data_dir / fname, 'wb') as dst:
                dst.write(src.read())
    print('Done.')

download_movielens(DATA_DIR)

Already downloaded.


### Step 2 — Load & Build Item Text

In [3]:
movies = pd.read_csv(
    DATA_DIR / 'movies.dat',
    sep='::',
    engine='python',
    names=['movie_id', 'title', 'genres'],
    encoding='latin-1'
)

movies['year']        = movies['title'].str.extract(r'\((\d{4})\)').astype(float)
movies['title_clean'] = movies['title'].str.replace(r'\s*\(\d{4}\)', '', regex=True).str.strip()
movies['genres_clean']= movies['genres'].str.replace('|', ' ', regex=False)

# text to embed: title + genres
movies['text'] = movies['title_clean'] + ' | ' + movies['genres_clean']

print(f'Movies: {len(movies)}')
movies[['title_clean', 'genres_clean', 'text']].head(5)

Movies: 3883


,title_clean,genres_clean,text
0,Toy Story,Animation Children's Comedy,Toy Story | Animation Children's Comedy
1,Jumanji,Adventure Children's Fantasy,Jumanji | Adventure Children's Fantasy
2,Grumpier Old Men,Comedy Romance,Grumpier Old Men | Comedy Romance
3,Waiting to Exhale,Comedy Drama,Waiting to Exhale | Comedy Drama
4,Father of the Bride Part II,Comedy,Father of the Bride Part II | Comedy


### Step 3 — Embed

In [4]:
EMB_PATH = DATA_DIR / 'embeddings.npy'

if EMB_PATH.exists():
    print('Loading cached embeddings ...')
    embeddings = np.load(EMB_PATH)
else:
    print(f'Encoding with {SBERT_MODEL} ...')
    model = SentenceTransformer(SBERT_MODEL)
    embeddings = model.encode(
        movies['text'].tolist(),
        batch_size=256,
        show_progress_bar=True,
        normalize_embeddings=True   # required for cosine similarity
    )
    np.save(EMB_PATH, embeddings)
    print('Saved to cache.')

print(f'Shape: {embeddings.shape}')  # expected shape: (3883, 384)

Loading cached embeddings ...
Shape: (3883, 384)


### Step 4 — Save movies dataframe

In [5]:
movies.to_pickle(DATA_DIR / 'movies.pkl')
print('Saved: data/movies.pkl')
print('Saved: data/embeddings.npy')
print()


Saved: data/movies.pkl
Saved: data/embeddings.npy

